In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt



In [5]:
# Load frozen Phase 1 input
adata_full = sc.read_h5ad(
    "../data/processed/Filbin_cohort_raw.h5ad"
    )
print(f"Loaded: {adata_full.shape[0]} cells x {adata_full.shape[1]} genes")
print(f"X dtype: {adata_full.X.dtype}")
print(f"Layers: {list(adata_full.layers.keys())}")

Loaded: 4058 cells x 23686 genes
X dtype: float64
Layers: ['tpm']


In [6]:
# Subset to Phase 1 cohort (paper-exact 2,458 cells)
adata = adata_full[adata_full.obs["in_paper_cohort"]].copy()
del adata_full  # free memory

In [7]:
# Cast X and the tpm layer to float32
adata.X = adata.X.astype(np.float32)
adata.layers["tpm"] = adata.layers["tpm"].astype(np.float32)

In [9]:
# Anchor verification
n_cells, n_genes = adata.shape
print(f"\nFilbin cohort: {n_cells} cells x {n_genes} genes")
print(f"Expected: 2458 cells x 23686 genes")
print(f"Anchor match: {n_cells == 2458 and n_genes == 23686}")


Filbin cohort: 2458 cells x 23686 genes
Expected: 2458 cells x 23686 genes
Anchor match: True


In [ ]:
# Per-tumor cell counts (sanity check)
print(f"\nCells per tumor:")
print(adata.obs["patient"].value_counts().sort_index())


Cells per tumor:
BCH836     527
BCH869     492
BCH1126    299
MUV1       146
MUV5       708
MUV10      286
Name: patient, dtype: int64


In [24]:
# Confirm X is still raw TPM at this point (per-cell sum should be ~1e6)
per_cell_sums = np.asarray(adata.X.sum(axis=1)).flatten()
print(f"\nMedian per-cell TPM sum: {np.median(per_cell_sums):.1f}  (expect 1,000,000)")


Median per-cell TPM sum: 1000000.0  (expect 1,000,000)


In [25]:
# --- Substep 3c: compute Ea from raw TPM, identify genes to keep ---
# Ea_i = log2(mean(TPM_i,:) + 1), per Filbin SM.
# Computed on the RAW TPM layer, not on log-transformed data.
tpm = adata.layers["tpm"]                       # (cells, genes), float32
mean_tpm_per_gene = np.asarray(tpm.mean(axis=0)).flatten()
Ea = np.log2(mean_tpm_per_gene + 1.0)

# Store Ea in var so it persists and can be re-inspected
adata.var["Ea"] = Ea

# Filter: drop genes with Ea < 4
keep_genes = Ea >= 4
n_keep = int(keep_genes.sum())
n_drop = int((~keep_genes).sum())
print(f"Genes retained (Ea >= 4): {n_keep}")
print(f"Genes dropped  (Ea <  4): {n_drop}")
print(f"Total:                    {n_keep + n_drop}  (expect 23686)")

# --- Substep 3b: apply Filbin transform to .X (still on full gene set) ---
# E = log2(TPM/10 + 1). /10 rationale: TPM normalizes to 1e6 transcripts,
# but single cells contain ~1e5 transcripts; /10 aligns the +1 pseudocount
# with the actual single-cell detection limit. Suva-lab convention.
adata.X = np.log2(adata.layers["tpm"] / 10.0 + 1.0).astype(np.float32)

# Quick sanity: E values should be non-negative, finite, modest range
X_arr = adata.X
print(f"\nE matrix dtype: {X_arr.dtype}")
print(f"E min/max:      {X_arr.min():.3f} / {X_arr.max():.3f}")
print(f"E mean (overall): {X_arr.mean():.3f}")
print(f"Fraction of E values == 0: {(X_arr == 0).mean():.3f}  (expect ~0.76 — matches raw zero fraction)")

# --- Now apply the gene filter to both .X and layers["tpm"] ---
adata = adata[:, keep_genes].copy()
print(f"\nAfter gene filter: {adata.shape[0]} cells x {adata.shape[1]} genes")

Genes retained (Ea >= 4): 8494
Genes dropped  (Ea <  4): 15192
Total:                    23686  (expect 23686)

E matrix dtype: float32
E min/max:      0.000 / 15.150
E mean (overall): 0.595
Fraction of E values == 0: 0.775  (expect ~0.76 — matches raw zero fraction)

After gene filter: 2458 cells x 8494 genes


### per tumor centralization | save `Er` in a different layer

In [26]:
# Per-tumor centering: Er_ij = E_ij - mean(E_i across cells of cell j's tumor)
# Stored in adata.layers["centered"]; .X (= E) is untouched.

# Start from a copy of .X so we don't mutate it during the operation.
Er = adata.X.copy()

# Group cell indices by patient (this is "tumor" in Filbin's wording).
# Using groupby on obs returns row positions per group — no Python loop over cells.
for patient, idx in adata.obs.groupby("patient", observed=True).indices.items():
    tumor_mean = Er[idx, :].mean(axis=0)        # per-gene mean within this tumor
    Er[idx, :] = Er[idx, :] - tumor_mean        # broadcast-subtract

adata.layers["centered"] = Er.astype(np.float32)

# Sanity check: per-tumor, per-gene column means of the centered matrix
# should be ~0 (up to float32 rounding, ~1e-6 to 1e-7).
print("Per-tumor max |column mean| of centered matrix (should be ~0):")
for patient, idx in adata.obs.groupby("patient", observed=True).indices.items():
    col_means = adata.layers["centered"][idx, :].mean(axis=0)
    print(f"  {patient:8s}  n={len(idx):4d}  max|mean| = {np.abs(col_means).max():.2e}")

# Quick distributional check: centered values should be roughly symmetric around 0,
# with a long positive tail (high expressors in their tumor) and a clipped negative
# tail (since the floor of E is 0, the floor of Er per tumor is -mean_tumor).
Er_arr = adata.layers["centered"]
print(f"\nEr min/max:  {Er_arr.min():.3f} / {Er_arr.max():.3f}")
print(f"Er mean:     {Er_arr.mean():.3e}   (should be ~0 overall by construction)")
print(f"Er std:      {Er_arr.std():.3f}")

Per-tumor max |column mean| of centered matrix (should be ~0):
  MUV1      n= 146  max|mean| = 3.48e-06
  MUV5      n= 708  max|mean| = 9.09e-06
  MUV10     n= 286  max|mean| = 4.39e-06
  BCH836    n= 527  max|mean| = 8.36e-06
  BCH869    n= 492  max|mean| = 6.05e-06
  BCH1126   n= 299  max|mean| = 5.18e-06

Er min/max:  -9.322 / 15.100
Er mean:     1.187e-08   (should be ~0 overall by construction)
Er std:      1.641


In [30]:
# Freeze Step 3 output
out_path = "../data/processed/Filbin_step3.h5ad"
adata.write_h5ad(out_path, compression="gzip")

import os
size_mb = os.path.getsize(out_path) / 1024**2
print(f"Wrote {out_path}  ({size_mb:.1f} MB)")

# Round-trip verification: load it back, confirm anchors
adata_check = sc.read_h5ad(out_path)
print(f"\nRound-trip check:")
print(f"  shape:           {adata_check.shape}  (expect 2458 x 8494)")
print(f"  layers:          {list(adata_check.layers.keys())}  (expect tpm, centered)")
print(f"  patients:        {sorted(adata_check.obs['patient'].unique())}")
print(f"  X mean:          {adata_check.X.mean():.4f}  (expect a value higher than 0.5953)")
print(f"  centered mean:   {adata_check.layers['centered'].mean():.2e}  (expect ~0)")
print(f"  Ea min in var:   {adata_check.var['Ea'].min():.3f}  (expect >= 4.0)")

Wrote ../data/processed/Filbin_step3.h5ad  (121.8 MB)

Round-trip check:
  shape:           (2458, 8494)  (expect 2458 x 8494)
  layers:          ['centered', 'tpm']  (expect tpm, centered)
  patients:        ['BCH1126', 'BCH836', 'BCH869', 'MUV1', 'MUV10', 'MUV5']
  X mean:          1.4282  (expect a value higher than 0.5953)
  centered mean:   1.19e-08  (expect ~0)
  Ea min in var:   4.000  (expect >= 4.0)


In [32]:
print(f"Mean E on post-filter matrix (8494 genes): {adata.X.mean():.4f}")
print(f"Frac zeros on post-filter matrix:          {(adata.X == 0).mean():.3f}")

Mean E on post-filter matrix (8494 genes): 1.4282
Frac zeros on post-filter matrix:          0.544


# Step 3 — Transform, gene filter, per-tumor centering

**Inputs:** `data/processed/Filbin_cohort_raw.h5ad` (4,058 cells × 23,686 genes; raw TPM).
**Output:** `data/processed/Filbin_step3.h5ad` (2,458 cells × 8,494 genes; three layers).
**Purpose:** prepare the working matrix that all subsequent analyses (cell–cell correlation, NMF, signature scoring) will operate on.

---

## What Step 3 does, in one paragraph

Take the frozen Phase 1 cohort, log-transform TPM into expression values `E` with the Suvà-lab convention, drop genes that are too lowly expressed to carry analyzable signal across the cohort, and remove each tumor's mean expression profile so that downstream analyses see *within-tumor* heterogeneity rather than *between-tumor* differences. The output is the matrix Filbin's pipeline actually runs on from this point forward.

---

## Substep 3a — Load, cast, subset

Loaded the frozen file (all 4,058 cells with full metadata), subset to the Phase 1 cohort via the pre-computed `in_paper_cohort` boolean tag, and cast `.X` and `layers["tpm"]` from float64 to float32.

**Why subset by the frozen tag rather than re-deriving cohort logic?** The tag was computed once at the end of Step 2 with the paper-exact definition (6 primary tumors, no model cells, no MGH66/101/104). Re-deriving it here would create a second source of truth and a drift hazard. Trust the tag, verify the count.

**Why float32?** TPM values from RSEM carry ~2–3 significant digits of biological precision; float32 has ~7 decimal digits of representational precision. Float64 is overkill by 4 orders of magnitude. Halving memory matters because the centering step temporarily holds two copies of the matrix.

**Anchor checks passed:**
- Cohort size: **2,458 cells** ✓ (paper-exact, locked invariant)
- Six patients: MUV1, MUV5, MUV10, BCH836, BCH869, BCH1126 ✓
- BCH869 primary tissue: **492 cells** ✓ (locked invariant)
- Per-cell TPM sum: ~1,000,000 ✓ (confirms `.X` is still raw TPM at this point — not silently transformed by an earlier step)

**Per-tumor cell counts:** MUV5 (708), BCH836 (527), BCH869 (492), BCH1126 (299), MUV10 (286), MUV1 (146). Sum = 2,458 ✓.

**Note for downstream:** MUV5 is ~5× larger than MUV1. Per-tumor centering and per-tumor NMF (Step 7) are robust to this imbalance because each tumor is treated independently. But any cross-tumor pooled analysis (cell–cell correlation distributions in Step 4, lineage-score distributions in Step 9) implicitly weights MUV5 most. Flagged.

---

## Substep 3b — Filbin's transform: `E = log2(TPM/10 + 1)`

Applied the Suvà-lab transform to `.X`, preserved raw TPM in `layers["tpm"]`.

**Why `log2`?** Standard for expression data. Gene expression spans 4–5 orders of magnitude; log compresses this into a range that respects relative (fold) differences and tames the influence of high-expressors on means, correlations, and clustering distances.

**Why `+1` pseudocount?** Without it, `log(0)` is undefined and `log(small)` produces large-magnitude negatives that dominate distance calculations from cells where a gene is simply undetected. The `+1` maps zero-TPM to zero-E exactly (`log2(0+1) = 0`), making "not detected" sit at the floor of the scale.

**Why divide TPM by 10?** This is the distinctive Suvà-lab choice (Tirosh 2016, Venteicher 2017, Filbin 2018), and it's worth understanding. TPM normalizes each cell's expression vector to sum to 1,000,000 transcripts, but a real single cell contains on the order of 100,000 transcripts (estimated from RNA content). TPM therefore inflates each transcript's count by a factor of ~10. Without dividing by 10, the `+1` pseudocount becomes negligible relative to the inflated TPM scale, and the transform behaves as if every dropout event were a confident "very low expression" measurement. Dividing by 10 first restores the pseudocount to the right order of magnitude relative to the actual single-cell detection limit — so a TPM value of 10 (the rough sensitivity floor) becomes `log2(1 + 1) = 1`, distinguishable from but not vastly different from zero.

**Anchor checks passed:**
- E min/max: **0.000 / 15.150** — floor is zero exactly (transform of TPM=0), max corresponds to TPM ≈ 365,000 (a high-expressor saturating in some cells).
- Fraction of zeros in E: **0.775** ≈ fraction of zeros in raw TPM (0.765) — transform sends zeros to zeros exactly, so this is the expected sanity check.

---

## Substep 3c — Gene filter: drop `Ea < 4`

Computed `Ea(i) = log2(mean(TPM_i,:) + 1)` per gene on the **raw TPM layer** (not on E), then dropped genes with `Ea < 4`.

**Important order-of-operations note.** `Ea` is defined on `mean(TPM)`, not on `mean(E)`. Because of Jensen's inequality, `mean(log2(TPM/10+1)) ≠ log2(mean(TPM)+1)` — they're different quantities. We computed `Ea` from `layers["tpm"]` (raw, never log-transformed), which is the unambiguously correct source. This is a common foot-gun in replications.

**What does `Ea ≥ 4` mean biologically?** `Ea ≥ 4` ⇒ `mean(TPM) ≥ 2^4 − 1 = 15`. So we kept genes whose *cohort-average* TPM is at least ~15. Genes below this floor are detected too rarely or too weakly across the 2,458 cells to support meaningful cross-cell analysis: their per-cell signal is dominated by dropout noise. Filbin's filter is conservative — it explicitly trades sensitivity (losing some real but lowly-expressed signal) for robustness (excluding genes that would inject noise into correlation/NMF).

**Outcome:** retained **8,494 genes** (dropped 15,192 of 23,686). This is a measured outcome, not a paper-stated target — Filbin's SM gives "~5,300 genes per cell on average" but doesn't state a cohort-level retained-gene count for the H3K27M data. The often-quoted "8,008 genes" is from the Tirosh 2016 oligodendroglioma paper (same filter rule, different dataset). Our 8,494 is in the same ballpark, consistent with Filbin's H3K27M cohort detecting slightly more genes per cell than Tirosh's oligodendroglioma cohort.

**Why no separate variable-gene (HVG) selection step?** Modern Scanpy/Seurat tutorials typically follow normalization with an HVG step (top ~2,000 most variable genes) before PCA. Filbin does not. All downstream methods in the paper — cell–cell correlation, hierarchical clustering, PCA, NNMF — operate on the full set of "analyzed genes" (post `Ea < 4`). The Ea filter *is* the feature-selection step. HVG selection deferred to Phase 2 (modernization item M2).

---

## Substep 3d — Per-tumor centering: `Er = E − mean(E_within_tumor)`

For each patient (tumor), computed the per-gene mean across that tumor's cells and subtracted it from those cells. Stored result in `adata.layers["centered"]`; `adata.X` retains the un-centered `E` for downstream methods that expect log-normalized non-centered input (e.g., `sc.tl.score_genes` in Step 5).

**Why centering at all?** Cell–cell correlation, PCA, and NMF all respond to *variation* in the data. If a gene has the same value in every cell, it carries no signal for these methods regardless of whether that value is high or low. Centering removes the per-gene offset, leaving variation around the mean as the working signal.

**Why per-tumor rather than global?** This is the methodological pivot of the entire paper. The Step 1 preliminary UMAP showed raw data is dominated by **patient of origin**, not cell type. Each tumor has its own technical baseline (library complexity, batch effects) and its own biological baseline (driver mutations, average expression profile of that tumor's cells). Global centering — subtracting each gene's cohort-wide mean — removes the cohort baseline but leaves the per-tumor offsets intact. A gene that's systematically higher in BCH869 than in MUV1 would still look "high" in BCH869 cells after global centering, and would still dominate any clustering. **Per-tumor centering removes each tumor's baseline independently**, so what remains is the variation *within* each tumor: which cells in MUV5 are cycling-high, which cells in BCH869 are OC-like, etc. This is exactly the question Filbin wants to ask, and it is why the paper sidesteps the patient-effect batch problem without ever doing batch correction.

**What we lose by per-tumor centering.** The centered matrix can no longer answer questions like "is PDGFRA higher in BCH869 than in MUV1 on average?" — every tumor's per-gene mean is zero by construction. Cross-tumor *average* comparisons must be made from `E` (in `.X`) or raw TPM (in `layers["tpm"]`). The centered matrix is purposefully built to answer *intra-tumor* questions, and *only* those.

**Anchor checks passed:**
- Per-tumor `max|column mean|` of centered matrix: 3.5e-6 to 9.1e-6 across tumors — float32 numerical residual, scales with √n as expected. Centering is mathematically correct.
- Overall Er mean: 1.2e-8 — essentially zero (sum of per-tumor zero-mean groups).
- Er range: [−9.32, +15.10]. Negative tail floor is set by `−tumor_mean` at undetected positions of high-mean genes; positive tail max ≈ original E max (a strong outlier dominates its tumor mean only weakly).
- Er std = 1.64 — larger than typical Scanpy log-normalized data (~0.5–1.0) because the Filbin `log2(TPM/10+1)` transform produces larger E values than Scanpy's `log1p(normalize_total)` default.

---

## Output: `Filbin_step3.h5ad`

| Component | Shape | Content | Used downstream by |
|---|---|---|---|
| `adata.X` | 2458 × 8494 | `E = log2(TPM/10+1)`, log-transformed, **not centered** | Signature scoring (Step 5, 9); anywhere a log-normalized matrix is expected |
| `adata.layers["tpm"]` | 2458 × 8494 | Raw TPM, gene-filtered subset | Recovery; sanity checks; recomputing Ea-style quantities |
| `adata.layers["centered"]` | 2458 × 8494 | `Er = E − mean(E within tumor)` | Cell–cell correlation (Step 4); per-tumor NMF (Step 7) |
| `adata.var["Ea"]` | 8494 | Aggregate expression per retained gene | Sanity checks; all values ≥ 4 by construction |
| `adata.obs` | 2458 rows | Cohort metadata: patient, sample, source, cohort tags, QC scores | Per-tumor grouping; downstream cell labels |

File size: **121.8 MB** (gzip-compressed). Compression exploits the ~54% zero fraction in the gene-filtered matrix.

---

## What we did **not** do, and why (for Phase 1)

- **No HVG / variable-gene selection.** Filbin doesn't. Modernization deferred to M2.
- **No `sc.pp.normalize_total` / `sc.pp.log1p`.** Those are for raw UMI counts (10x). Our data is already TPM-normalized by RSEM, and Filbin's transform is `log2(TPM/10+1)`, not `log1p(median-normalized counts)`.
- **No additional cell QC.** Done diagnostically in Step 2 and locked-in by the 2,458 paper anchor.
- **No batch correction (Combat, Harmony, scVI).** The per-tumor centering *is* Filbin's response to the patient-of-origin batch effect, and it is sufficient given the analyses that follow (per-tumor NMF in Step 7, signature scoring in Steps 5/9 — neither of which co-embeds cells across tumors). Cross-dataset integration is a Phase 2 question (M6).

---

## Methodological note for the thesis: why this preprocessing matters

Three choices in Step 3 are load-bearing for everything that follows. They will recur in the thesis methods section:

1. **The `/10` in the transform** is what makes the +1 pseudocount meaningful for single-cell data. Without it, the transform behaves as if every dropout were a confident low-expression measurement.

2. **The `Ea < 4` filter** trades sensitivity for robustness. Lowly-expressed genes inject dropout noise into correlation and NMF; the filter removes that noise floor at the cost of losing some real signal in rare-marker genes. This is a deliberate methodological choice consistent with a "discover robust programs, not rare events" research goal.

3. **Per-tumor centering** is the paper's batch-effect strategy. By removing each tumor's mean, the analyses that follow ask "what varies within tumors?" rather than "what differs between tumors?" — and the answers (lineage programs, cell cycle, OPC-like states) turn out to recur across tumors, which is itself the central finding.